# FAST-UAV - Off-design mission performance
*Author: Félix Pollet - 2023* <br>

ㅊ이 노트북은 모든 작전 임무에 대해 사전 정의된 UAV 설계의 성능을 계산하는 방법에 대한 예제를 제공합니다.

이를 위해 이전 설계 최적화 프로세스의 결과를 [problem_outputs_DJI_M600_mdo.xml](../data/problem_outputs_DJI_M600_mdo.xml) 파일에 저장된 결과를 재사용합니다. 현재 문제는 디자인 최적화에 사용되는 문제와는 약간 다릅니다. 실제로 성능 모듈에 대한 간단한 평가로 구성되어 있습니다. 설계가 이미 알려져 있거나 수정되었기 때문에 어떤 종류의 최적화도 수행되지 않습니다.

In [54]:
# Import required librairies
import os.path as pth
import openmdao.api as om
import logging
import warnings
import shutil
import fastoad.api as oad
from time import time
import matplotlib.pyplot as plt
from fastuav.utils.postprocessing.analysis_and_plots import *

# Declare paths to folders and files
DATA_FOLDER_PATH = "./data"
WORK_FOLDER_PATH = "./workdir"
CONFIGURATION_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "configurations")
SOURCE_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "source_files")

CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "multirotor_performance.yaml")  # The problem is now a simple evaluation of the UAV performance, not an optimization
# CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "multirotor_mdo.yaml") # You may provide a different configuration file

SOURCE_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_outputs_DJI_M600_mdo.xml")  # The source file is the output file of a previous design optimization process.

# For having log messages display on screen
#logging.basicConfig(level=logging.INFO, format="%(levelname)-8s: %(message)s")
#warnings.filterwarnings(action="ignore")

# For using all screen width
from IPython.display import display, HTML, IFrame
display(HTML("<style>.container { width:95% !important; }</style>"))

## 1. Setting up the performance analysis

멀티로터_퍼포먼스.yaml](./데이터/구성/멀티로터_퍼포먼스.yaml) 파일을 살펴보세요. 성능 분석에 사용되는 모델은 `Missions` 모듈로만 구성되어 있습니다. 구성 요소에 대한 설계 모델은 사용되지 않고 성능 모델만 사용됩니다.<br> <br>

```yaml
model:
    missions:
        id: fastuav.performance.mission
        file_path: ../missions/missions_multirotor.yaml
```
임무, 경로 및 비행 단계의 정의는 구성 파일에 경로가 제공된 [임무 파일](./data/missions/missions_multirotor.yaml)에 정의되어 있습니다. 임무는 경로(예: 주 경로와 우회 비행)의 집합으로 정의되며, 이는 다시 비행 단계의 집합으로 정의됩니다. 각 비행 단계는 이를 설명하는 `phase_id`로 식별됩니다. 멀티로터 드론과 관련된 위상 식별자는 '수직 상승', '멀티로터 순항' 및 '호버'입니다.

In [46]:
N2_FILE = pth.join(WORK_FOLDER_PATH, "n2.html")
oad.write_n2(CONFIGURATION_FILE, N2_FILE, overwrite=True)
from IPython.display import IFrame
IFrame(src=N2_FILE, width="100%", height="500px")

## 2. Run the performance analysis
성능 분석을 실행하려면 드론 설계를 설명하는 XML 파일이 필요합니다. 앞서 설명한 대로 설계 문제의 결과물일 수 있습니다([멀티로터 설계 노트북](./1_Multirotor_Design.ipynb) 참조).

문제에 대한 입력 파일을 생성해 보겠습니다:

In [55]:
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)

The following variables have NaN values: ['mission:operational_1:diversion:payload:mass', 'mission:operational_1:diversion:cruise:altitude', 'mission:operational_1:diversion:hover:duration', 'mission:operational_1:dISA', 'mission:operational_1:diversion:hover:payload:power', 'mission:operational_1:diversion:cruise:distance', 'mission:operational_1:diversion:cruise:speed', 'mission:operational_1:diversion:cruise:payload:power', 'mission:operational_1:main_route:payload:mass', 'mission:operational_1:main_route:cruise:altitude', 'mission:operational_1:main_route:hover:duration', 'mission:operational_1:main_route:hover:payload:power', 'mission:operational_1:main_route:takeoff:altitude', 'mission:operational_1:main_route:climb:rate', 'mission:operational_1:main_route:climb:speed', 'mission:operational_1:main_route:climb:payload:power', 'mission:operational_1:main_route:cruise:distance', 'mission:operational_1:main_route:cruise:speed', 'mission:operational_1:main_route:cruise:payload:power',

'D:/KangDH/fast_UAV/FAST-UAV/src/fastuav/notebooks/workdir/problem_inputs.xml'

경고 메시지는 미션 프로필과 관련된 일부 변수에 아직 값이 할당되지 않았음을 나타냅니다. 변수 뷰어를 사용하여 평가하려는 미션의 값을 입력합니다. 예를 들어, 첫 번째 운영 임무는 고도 150미터, 비행 속도 15m/s에서 페이로드 질량 3kg(설계 페이로드 질량 5.5 대신)의 순항 거리 10,000미터로 구성될 수 있습니다.

In [50]:
INPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_inputs.xml")
oad.variable_viewer(INPUT_FILE)

'미션' 메뉴에서 '변수 뷰어'를 사용하여 미션 프로필의 값을 입력할 수 있습니다(저장하는 것을 잊지 마세요!).

완료되면 `evaluate_problem`...을 사용하여 문제를 실행할 수 있습니다.

In [51]:
eval_problem = oad.evaluate_problem(CONFIGURATION_FILE, overwrite=True)

... and have a look at the results.

In [52]:
OUTPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_outputs.xml")
oad.variable_viewer(OUTPUT_FILE)

## 3. Analysis and plots
현재 포스트 프로세싱 플롯은 미션에서 소비된 에너지를 시각화하는 것으로 제한되어 있습니다. 여러 미션이 계산된 경우 동일한 플롯에서 비교할 수 있습니다.

In [53]:
mission_name = "operational_1"  # name of the mission to plot
fig = energy_breakdown_sun_plot_drone(OUTPUT_FILE, mission_name=mission_name)
fig.show()

ValueError: 'mission:operational_1:energy' is not in list

In [9]:
mission_name = "operational_2"  # secondary mission
fig = energy_breakdown_sun_plot_drone(OUTPUT_FILE, mission_name=mission_name, fig=fig)
fig.show()

NameError: name 'fig' is not defined